### Coding tasks

#### Task 1: Load and summarise a sensor session

##### Prompt: 'Load session_clean.csv. Give me a summary: how many rows, what activities are present, what percentage of data each activity covers, and the mean accelerometer magnitude per activity.'


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv('session_clean.csv', parse_dates=['timestamp'])

# summary
print(f'Shape: {df.shape}')
print(f'Total rows: {len(df)}')
print(f'Duration: From {df.timestamp.min()} to {df.timestamp.max()}')
print(f'Top 5 summary:\n{df.head(5)}')

# derived feature
df['acc_magnitude'] = np.sqrt(df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2)

summary = df.groupby('activity').aggregate(
    n_rows = ('acc_x', 'count'),
    pct = ('acc_x', lambda x: round(len(x) / len(df) *100, 1)),
    acc_mean = ('acc_magnitude', 'mean'),
    acc_std = ('acc_magnitude', 'std'),
).reset_index()

print(f'\nSummary:\n{summary}')

Shape: (3000, 10)
Total rows: 3000
Duration: From 2024-03-01 09:00:00 to 2024-03-01 09:00:29.990000
Top 5 summary:
                timestamp     acc_x     acc_y     acc_z    gyro_x    gyro_y  \
0 2024-03-01 09:00:00.000  0.348357 -1.003904  9.465776  0.015308  0.023496   
1 2024-03-01 09:00:00.010  0.030868 -0.480193  9.610721  0.021468 -0.037580   
2 2024-03-01 09:00:00.020  0.423844 -0.256803  9.517382  0.009974 -0.006556   
3 2024-03-01 09:00:00.030  0.861515  0.893844  9.635601 -0.038850 -0.000833   
4 2024-03-01 09:00:00.040 -0.017077  0.228277  9.735755 -0.003108  0.000318   

     gyro_z participant_id session_id activity  
0 -0.002868           P001       S001  walking  
1 -0.000653           P001       S001  walking  
2  0.001286           P001       S001  walking  
3  0.018937           P001       S001  walking  
4 -0.014944           P001       S001  walking  

Summary:
   activity  n_rows   pct  acc_mean   acc_std
0   running     800  26.7  9.835197  0.315198
1   sitting   

#### Task 2: Validate a dirty dataset — find ALL issues

##### Prompt: 'Load session_dirty.csv. Write a function that checks for and reports all data quality issues.'

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv('session_dirty.csv', parse_dates=['timestamp'])

print(f'Shape: {df.shape}')
print(f'Duration: From {df.timestamp.min()} to {df.timestamp.max()}')
print(f'Top 5 Summary:\n{df.head(5)}')

Shape: (3001, 10)
Duration: From 2024-02-28 08:00:00 to 2024-03-01 09:00:29.990000
Top 5 Summary:
                timestamp     acc_x     acc_y     acc_z    gyro_x    gyro_y  \
0 2024-03-01 09:00:00.000  0.348357 -1.003904  9.465776  0.015308  0.023496   
1 2024-03-01 09:00:00.010  0.030868 -0.480193  9.610721  0.021468 -0.037580   
2 2024-03-01 09:00:00.020  0.423844 -0.256803  9.517382  0.009974 -0.006556   
3 2024-03-01 09:00:00.030  0.861515  0.893844  9.635601 -0.038850 -0.000833   
4 2024-03-01 09:00:00.040 -0.017077  0.228277  9.735755 -0.003108  0.000318   

     gyro_z participant_id session_id activity  
0 -0.002868           P002       S002  walking  
1 -0.000653           P002       S002  walking  
2  0.001286           P002       S002  walking  
3  0.018937           P002       S002  walking  
4 -0.014944           P002       S002  walking  


In [14]:
def validate_sessions(df, session_id='unknown', acc_max=16.0, null_threshold=0.01):
    issues = []
    report = {'session_id': session_id, 'n_rows': len(df), 'issues': [], 'passed': False}

    # 1. required columns
    required = ['timestamp', 'acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z', 'activity']
    missing = [c for c in required if c not in df.columns]
    if missing:
        issues.append(f'Missing columns: {missing}')
    
    # 2. duplicate rows
    n_dupes = df.duplicated().sum()
    if n_dupes > 0:
        issues.append(f'Duplicate rows: {n_dupes}')

    # 3. null rates
    null_rates = df.isnull().mean()
    high_null = null_rates[null_rates > null_threshold]
    if not high_null.empty:
        issues.append(f'High null rate columns: {high_null.round(3).to_dict()}')
    
    # 4. timestamp monotonicity
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    if not df['timestamp'].is_monotonic_increasing:
        n_bad = (df['timestamp'] < df['timestamp'].shift(1)).sum()
        issues.append(f'Non-monotonic timestamps: {n_bad} rows')

    # 5. sensor clipping
    for col in ['acc_x', 'acc_y', 'acc_z']:
        if col in df.columns:
            clipped = (df[col].abs() > acc_max).sum()
            if clipped > 0:
                issues.append(f'Clipped samples in {col}: {clipped}')
    
    # 6. missing labels
    if 'activity' in df.columns:
        missing_labels = df['activity'].isnull().sum()
        if missing_labels > 0:
            issues.append(f'Missing activity labels: {missing_labels}')
    
    report['issues'] = issues
    report['passed'] = len(issues) == 0
    return report

df = pd.read_csv('session_dirty.csv')
result = validate_sessions(df, session_id='S002')
print(f"Passed: {result['passed']}")
for issue in result['issues']:
    print(f'  - {issue}')

Passed: False
  - Duplicate rows: 1
  - High null rate columns: {'acc_x': 0.013, 'acc_y': 0.013, 'activity': 0.017}
  - Non-monotonic timestamps: 2 rows
  - Clipped samples in acc_z: 15
  - Missing activity labels: 50


#### Compute rolling features

##### Prompt: 'For the clean session, compute a 50-sample rolling mean and standard deviation of accelerometer magnitude. Flag windows where the std > 2.0 as high-variability.'


In [25]:
import pandas as pd
import numpy as np

df = pd.read_csv('session_clean.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
df['acc_magnitude'] = np.sqrt(df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2)

WINDOW = 50
df['rolling_mean'] = df['acc_magnitude'].rolling(window=WINDOW, min_periods=WINDOW).mean()
df['rolling_std'] = df['acc_magnitude'].rolling(window=WINDOW, min_periods=WINDOW).std()
df['high_variability'] = df['rolling_std'] > 0.3
print(f'High Variability:\n{df["high_variability"]}')
print(f'High Variability counts:\n{df["high_variability"].value_counts()}')

High Variability:
0       False
1       False
2       False
3       False
4       False
        ...  
2995    False
2996    False
2997    False
2998    False
2999    False
Name: high_variability, Length: 3000, dtype: bool
High Variability counts:
high_variability
True     1783
False    1217
Name: count, dtype: int64


In [28]:
n_flagged = df['high_variability'].sum()
print(f'High Variability windows: {n_flagged} / {len(df)} ({n_flagged/len(df)*100:.1f})%')
print(df[['timestamp', 'acc_magnitude', 'rolling_mean', 'rolling_std', 'high_variability']].head(60))

High Variability windows: 1783 / 3000 (59.4)%
                 timestamp  acc_magnitude  rolling_mean  rolling_std  \
0  2024-03-01 09:00:00.000       9.525234           NaN          NaN   
1  2024-03-01 09:00:00.010       9.622759           NaN          NaN   
2  2024-03-01 09:00:00.020       9.530275           NaN          NaN   
3  2024-03-01 09:00:00.030       9.715245           NaN          NaN   
4  2024-03-01 09:00:00.040       9.738446           NaN          NaN   
5  2024-03-01 09:00:00.050      10.076755           NaN          NaN   
6  2024-03-01 09:00:00.060       9.746257           NaN          NaN   
7  2024-03-01 09:00:00.070       9.373110           NaN          NaN   
8  2024-03-01 09:00:00.080      10.154809           NaN          NaN   
9  2024-03-01 09:00:00.090       9.560354           NaN          NaN   
10 2024-03-01 09:00:00.100       9.783750           NaN          NaN   
11 2024-03-01 09:00:00.110       9.953737           NaN          NaN   
12 2024-03-01 09:0

#### Task 4: Sliding window segmentation

##### Prompt: 'Segment the clean session into 5-second windows (500 rows at 100Hz) with 50% overlap. Assign each window the majority activity label. Return a summary.'

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('session_clean.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
df['acc_magnitude'] = np.sqrt(df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2)

WINDOW_SIZE = 500
STRIDE = 250
FEATURE_COLS = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z', 'acc_magnitude']

windows = []
for start in range(0, len(df) - WINDOW_SIZE + 1, STRIDE):
    window = df.iloc[start : start + WINDOW_SIZE]

    # quality gate: skip windows with nulls
    if window[FEATURE_COLS].isnull().any().any():
        continue

    label = window['activity'].mode() [0] # majority vote

    windows.append({
        'window_start': window['timestamp'].iloc[0],
        'window_end': window['timestamp'].iloc[-1],
        'label': label,
        'acc_mean': window['acc_magnitude'].mean(),
        'acc_std': window['acc_magnitude'].std(),
        'n_rows': len(window),
    })

windows_df = pd.DataFrame(windows)
print(f'Total windows: {len(windows_df)}')
print(f'Windows:\n{windows_df}')
print(windows_df['label'].value_counts())

Total windows: 11
Windows:
              window_start              window_end     label  acc_mean  \
0  2024-03-01 09:00:00.000 2024-03-01 09:00:04.990   walking  9.819563   
1  2024-03-01 09:00:02.500 2024-03-01 09:00:07.490   walking  9.811704   
2  2024-03-01 09:00:05.000 2024-03-01 09:00:09.990   walking  9.816480   
3  2024-03-01 09:00:07.500 2024-03-01 09:00:12.490   sitting  9.819626   
4  2024-03-01 09:00:10.000 2024-03-01 09:00:14.990   sitting  9.829012   
5  2024-03-01 09:00:12.500 2024-03-01 09:00:17.490   sitting  9.824539   
6  2024-03-01 09:00:15.000 2024-03-01 09:00:19.990   running  9.835245   
7  2024-03-01 09:00:17.500 2024-03-01 09:00:22.490   running  9.833695   
8  2024-03-01 09:00:20.000 2024-03-01 09:00:24.990   running  9.823196   
9  2024-03-01 09:00:22.500 2024-03-01 09:00:27.490  standing  9.843724   
10 2024-03-01 09:00:25.000 2024-03-01 09:00:29.990  standing  9.839807   

     acc_std  n_rows  
0   0.297195     500  
1   0.304010     500  
2   0.316126   

#### Task 5: Participant level train/test split

##### Prompt: 'Split the multi-participant campaign into 70% train, 10% validation, 20% test — at the participant level. Validate there is zero participant overlap between splits.'

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('campaign_multiparticipant.csv', parse_dates=['timestamp'])
df.head(5)

participants = df['participant_id'].unique().tolist()
print(f'Unique participants: {participants}')
np.random.seed(100)
np.random.shuffle(participants)
print(f'Unique participants: {participants}')

Unique participants: ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008', 'P009', 'P010', 'P011', 'P012', 'P013', 'P014', 'P015', 'P016', 'P017', 'P018', 'P019', 'P020']
Unique participants: ['P018', 'P020', 'P012', 'P019', 'P014', 'P007', 'P017', 'P002', 'P010', 'P015', 'P013', 'P006', 'P003', 'P005', 'P011', 'P001', 'P016', 'P008', 'P004', 'P009']


In [8]:
n = len(participants)
n_train = int(n*0.7)
n_val = int(n*0.1)

train_pids = set(participants[: n_train])
val_pids = set(participants[n_train : n_train + n_val])
test_pids = set(participants[n_train + n_val : ])

# validation
assert len(train_pids & val_pids) == 0, 'Train/val participant overlap!'
assert len(train_pids & test_pids) == 0, 'Train/test participant overlap!'
assert len(val_pids & test_pids) == 0, 'Val/test participant overlap!'

train_df = df[df['participant_id'].isin(train_pids)]
val_df = df[df['participant_id'].isin(val_pids)]
test_df = df[df['participant_id'].isin(test_pids)]

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name}: {split["participant_id"].nunique()} participants, {len(split)} rows')
    print(f'    Class distribution: {split["activity"].value_counts(normalize=True).round(2).to_dict()}')

Train: 14 participants, 31359 rows
    Class distribution: {'stairs_down': 0.23, 'walking': 0.19, 'stairs_up': 0.17, 'sitting': 0.17, 'running': 0.16, 'standing': 0.08}
Val: 2 participants, 4323 rows
    Class distribution: {'stairs_down': 0.55, 'standing': 0.17, 'running': 0.15, 'walking': 0.13}
Test: 4 participants, 8365 rows
    Class distribution: {'walking': 0.3, 'standing': 0.25, 'stairs_down': 0.18, 'stairs_up': 0.1, 'running': 0.1, 'sitting': 0.07}


#### Task 6: Motion artifact detection in PPG

##### Prompt: 'Load ppg_session.csv. Flag windows where accelerometer magnitude is high enough to corrupt the PPG signal. Summarise affected windows by activity.'


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('ppg_session.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
print(f"{df['participant_id'].unique().tolist()}")
print(f'Head:\n{df.head(10)}')

['P010']
Head:
                timestamp       ppg     acc_x     acc_y     acc_z  \
0 2024-03-01 10:00:00.000  1.106482  0.054918  0.022239  9.821172   
1 2024-03-01 10:00:00.040  1.089502 -0.034175  0.032498  9.757164   
2 2024-03-01 10:00:00.080  1.241597 -0.007397  0.003756  9.862518   
3 2024-03-01 10:00:00.120  1.264859  0.029796  0.033128  9.767078   
4 2024-03-01 10:00:00.160  1.340489 -0.099379 -0.003725  9.781024   
5 2024-03-01 10:00:00.200  1.300234  0.018781 -0.028936  9.773344   
6 2024-03-01 10:00:00.240  1.399933 -0.028213  0.010868  9.776177   
7 2024-03-01 10:00:00.280  1.293435 -0.031609 -0.006620  9.831279   
8 2024-03-01 10:00:00.320  1.209010 -0.021280  0.046738  9.859882   
9 2024-03-01 10:00:00.360  1.130977 -0.020881 -0.056970  9.788619   

   heart_rate_bpm activity participant_id session_id  
0       67.888160  resting           P010  S_PPG_001  
1       67.689729  resting           P010  S_PPG_001  
2       67.796095  resting           P010  S_PPG_001  
3    

In [11]:
# compute acc magnitude
df['acc_magnitude'] = np.sqrt(df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2)

# flag high motion windows
MOTION_THRESHOLD = 2.0
df['high_motion'] = df['acc_magnitude'] > MOTION_THRESHOLD

# rolling flag: if >30% of samples in 25-sample (1s) window are high motion,
df['motion_frac'] = df['high_motion'].rolling(25, min_periods=1).mean()
df['ppg_corrupted'] = df['motion_frac'] > 0.3

# summary by activity
summary = df.groupby('activity').agg(
    n_rows = ('ppg', 'count'),
    pct_corrupted = ('ppg_corrupted', 'mean'),
    mean_heart_rate = ('heart_rate_bpm', 'mean'),
).reset_index()
summary['pct_corrupted'] = (summary['pct_corrupted'] * 100).round(1)
print(summary.to_string(index=False))

activity  n_rows  pct_corrupted  mean_heart_rate
 resting    1200          100.0        68.075033
 running     800          100.0       144.521810


#### Task 7: Combine multiple CSV files from a folder

##### Prompt: 'You have a folder of session CSVs from a field collection campaign. Load them all, add a source_file column, combine into one DataFrame, and report which files had issues.'

In [14]:
import pandas as pd
import numpy as np
import glob
import os

def load_campaign_folder(folder_path):
    all_files = glob.glob(os.path.join(folder_path, '*.csv'))
    # print(all_files)
    if not all_files:
        raise ValueError(f'No CSV filed found in folder path: {folder_path}')
    
    loaded = []
    failed = []

    for path in all_files:
        fname = os.path.basename(path)
        # print(fname)
        try:
            df = pd.read_csv(path, parse_dates=['timestamp'])
            df['source_file'] = fname
            loaded.append(df)
        except Exception as e:
            failed.append({'file': fname, 'error': str(e)})
    
    print(f'Loaded: {len(loaded)} files, Failed: {len(failed)} files')
    if failed:
        for f in failed:
            print(f'    FAILED: {f["file"]} - {f["error"]}')
    
    if not loaded:
        raise RuntimeError('No files loaded successfully')

    combined = pd.concat(loaded, ignore_index=True)
    return combined, failed


In [15]:
df, errors = load_campaign_folder('./')
# print(df)
print(f'Combined: {len(df)} rows from {df["source_file"].nunique()} files')

Loaded: 4 files, Failed: 1 files
    FAILED: output.csv - Missing column provided to 'parse_dates': 'timestamp'
Combined: 52048 rows from 4 files


#### Task 8: End-to-end mini pipeline

##### Prompt: 'Write a pipeline that: loads a session, validates it, computes magnitude and rolling features, segments into windows, assigns labels, and outputs a summary CSV.'


In [22]:
import pandas as pd
import numpy as np

def run_pipeline(input_path, output_path, window_size=500, stride=250):
    print(f'[1/5] Loading {input_path}')
    df = pd.read_csv(input_path, parse_dates=['timestamp'])
    n_raw = len(df)

    print(f'[2/5] Validating - {n_raw} rows')
    issues = []
    if df.isnull().any().any():
        issues.append('Nulls detected - dropping')
    df = df.dropna(subset=['acc_x', 'acc_y', 'acc_x', 'activity'])
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.sort_values('timestamp').reset_index(drop=True)
    if not df['timestamp'].is_monotonic_increasing:
        issues.append('Non-monotonic timestamps detecting after sort')
    if issues:
        print(f'    Issues: {issues}')
    
    print(f'[3/5] Feature engineering - {len(df)} rows after cleaning')
    df['acc_magnitude'] = np.sqrt(df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2)
    df['rolling_mean'] = df['acc_magnitude'].rolling(50, min_periods=50).mean()
    df['rolling_std'] = df['acc_magnitude'].rolling(50, min_periods=50).std()

    print(f'[4/5] Windowing - wondow={window_size}, stride={stride}')
    feature_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z', 'acc_magnitude', 'rolling_mean', 'rolling_std']
    windows = []
    for start in range(0, len(df) - window_size + 1, stride):
        w = df.iloc[start: start + window_size]
        if w[feature_cols].isnull().any().any():
            continue
        
        windows.append({
            'window_id' : len(windows),
            'start_ts'  : w['timestamp'].iloc[0],
            'label'     : w['activity'].mode()[0],
            'acc_mean'  : w['acc_magnitude'].mean().round(3),
            'acc_std'   : w['acc_magnitude'].std().round(3),
            'roll_mean' : w['rolling_mean'].mean().round(3),
            'roll_std'  : w['rolling_std'].std().round(3),  
        })
    # print(windows)
    windows_df = pd.DataFrame(windows)

    print(f'[5/5] Writing {len(windows_df)} windows to {output_path}')
    windows_df.to_csv(output_path, index=False)
    print('Done.')
    print(windows_df['label'].value_counts().to_string())
    return windows_df

In [23]:
run_pipeline('./session_dirty.csv', './output.csv')

[1/5] Loading ./session_dirty.csv
[2/5] Validating - 3001 rows
    Issues: ['Nulls detected - dropping']
[3/5] Feature engineering - 2871 rows after cleaning
[4/5] Windowing - wondow=500, stride=250
[5/5] Writing 9 windows to ./output.csv
Done.
label
sitting     3
running     3
walking     2
standing    1


,window_id,start_ts,label,acc_mean,acc_std,roll_mean,roll_std
0,0,2024-03-01 09:00:02.640,walking,9.836,0.534,9.854,0.443
1,1,2024-03-01 09:00:05.290,walking,9.858,0.695,9.862,0.444
2,2,2024-03-01 09:00:07.900,sitting,9.880,0.823,9.884,0.505
3,3,2024-03-01 09:00:10.550,sitting,9.861,0.696,9.861,0.442
4,4,2024-03-01 09:00:13.160,sitting,9.844,0.534,9.840,0.338
5,5,2024-03-01 09:00:15.710,running,9.870,0.691,9.868,0.436
6,6,2024-03-01 09:00:18.330,running,9.910,0.931,9.910,0.544
7,7,2024-03-01 09:00:20.950,running,9.911,0.929,9.912,0.547
8,8,2024-03-01 09:00:23.510,standing,9.874,0.684,9.864,0.372
